# example / wasm / accumulate

**Intent**
- Estimates `wasm_out` from `wasm_in` and the state of the previous call, and must:
  - Scale the input by a fitted gain plus a fitted offset.
  - Give `wasm_*`, 28 check outputs, one family each of comparison, rounding, power and host maths.
  - Carry the running total and peak in `wasm_state_out`.
  - Exercise every tracer operator and all fifteen host maths functions in all three hosts.
  - Hold the rmse to 0.5, and each check output to 1e-9 against plain Python.
  - Stay a fixture, tested by `fab unittest` and `fab systest` and never released.

**Data**
- `training` is ten labelled points on a line with fixed noise, the only data the fit sees.
- `validation` is five more points, and `backtest` is ten further out, held back for acceptance.
- `golden` is six calls from 1 to 14 and back to 2, crossing both bounds and the ties of `nearest`.
- `golden` ends with a request with the input missing, which must give a signature error.
- The golden tolerance is 1e-9 for outputs that depend on a host maths function.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import numpy as np

from mlmodels import setup_workspace

ws = setup_workspace("example/wasm/accumulate")


ws.outputs[0] = ws.outputs[0]

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {ws.inputs[0]: xs, ws.outputs[0]: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


collect_run(ws, 1000, label="Ten labelled points on a line with fixed noise", name="wasm_accumulate_training", source="builder", build=lambda: labelled(0, 10))
collect_run(ws, 1000, label="Five more points on the line", name="wasm_accumulate_validation", source="builder", build=lambda: labelled(10, 5))
collect_run(ws, 1000, label="Ten further points on the line held back for acceptance", name="wasm_accumulate_backtest", source="builder", build=lambda: labelled(15, 10))

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "wasm_accumulate_training"), validation=ws.run(1000, "wasm_accumulate_validation"), backtest=ws.run(1000, "wasm_accumulate_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
gain, offset = np.polyfit(ws.training[ws.inputs[0]].to_numpy(), ws.training[ws.outputs[0]].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
ws.params = {"gain": GAIN, "offset": OFFSET}
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model, maths

GAIN, OFFSET = ws.params["gain"], ws.params["offset"]

LOW, HIGH = 5.0, 20.0
UNARY = ["acos", "asin", "atan", "cos", "cosh", "exp", "log", "log10", "log2", "sin", "sinh", "tan", "tanh"]
ws.params["unary"] = UNARY


def exercise(x, out):
    unit = maths.clip(x / 16.0, 0.0, 1.0)
    checks = {
        "bounded": maths.clip(out, LOW, HIGH),
        "band": maths.select(out < LOW, -1.0, maths.select(out > HIGH, 1.0, 0.0)),
        "band_inclusive": maths.select(out <= LOW, 0.0, maths.select(out >= HIGH, 2.0, 1.0)),
        "equal": maths.select(maths.eq(out, 8.5), 1.0, 0.0),
        "unequal": maths.select(maths.ne(out, 8.5), 1.0, 0.0),
        "floor": maths.floor(out),
        "ceil": maths.ceil(out),
        "trunc": maths.trunc(-out),
        "nearest": maths.nearest(out),
        "magnitude": maths.sqrt(abs(-out)),
        "polynomial": 0.01 * out**3 - 0.2 * out**2 + out,
        "power_const": out**1.5,
        "power_base": 2.0**unit,
        "power_unit": maths.power(unit, 2.5),
        "atan2": maths.atan2(out, x),
    }
    return checks | {name: getattr(maths, name)(unit) for name in UNARY}


def accumulate(x, total, peak):
    out = GAIN * x + OFFSET
    return [out, *exercise(x, out).values(), total + out, maths.maximum(peak, out)]


define_model(
    ws,
    model=1000,
    flavour="wasm",
    source=accumulate,
    label="Least squares line with every maths function and a running total and peak",
    constants={"GAIN": GAIN, "OFFSET": OFFSET, "LOW": LOW, "HIGH": HIGH, "UNARY": UNARY},
)

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models

GAIN, OFFSET = ws.params["gain"], ws.params["offset"]
UNARY = ws.params["unary"]
CHECKS = list(exercise(1.0, 3.5))
TOLERANT = [name for name in CHECKS if name in (*UNARY, "polynomial", "power_const", "power_base", "power_unit", "atan2")]


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        frame = getattr(ws, split)
        xs = frame[ws.inputs[0]].to_list()
        predicted = evaluate_model(ws, 1000, split)
        checks.append(
            {
                "split": split,
                "gate": split == "backtest",
                "output": ws.outputs[0],
                "name": "rmse",
                "value": evaluate_metrics(frame[ws.outputs[0]], predicted[ws.outputs[0]])["rmse"],
                "limit": 0.5,
                "direction": "max",
            }
        )
        for name in CHECKS:
            plain = [exercise(x, GAIN * x + OFFSET)[name] for x in xs]
            checks.append(
                {
                    "split": split,
                    "gate": split == "backtest",
                    "output": f"wasm_{name}",
                    "name": "rmse",
                    "value": evaluate_metrics(plain, predicted[f"wasm_{name}"])["rmse"],
                    "limit": 1e-9,
                    "direction": "max",
                }
            )
    return checks


ws.params["tolerant"] = TOLERANT
evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [1.0, 3.0, 8.0, 10.0, 14.0, 2.0, None], "expect": ["ok"] * 6 + ["error"]}
TOLERANT = ws.params["tolerant"]
package_model(ws, 1000, golden, target={"kind": "embedded", "language": "python"}, tolerance={f"wasm_{name}": {"absolute": 1e-9, "relative": 1e-9} for name in TOLERANT})